In [ ]:
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from scipy.stats import pearsonr, spearmanr

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

model_name = "distilbert-base-uncased"
dataset_name = "glue"
dataset_config = "stsb"
split_name = "validation"
device = "mps" if torch.backends.mps.is_available() else "cpu"
batch_size = 64 if device == "mps" else 32
max_length = 128
start_time = time.time()

print({
    "model_name": model_name,
    "dataset": f"{dataset_name}/{dataset_config}",
    "split": split_name,
    "device": device,
    "batch_size": batch_size,
    "max_length": max_length,
    "seed": seed,
})

In [ ]:
ds = load_dataset(dataset_name, dataset_config, split=split_name)
df = ds.to_pandas()[["sentence1", "sentence2", "label"]].copy()
df["label"] = df["label"].astype(np.float32)

print({"num_examples": len(df), "columns": df.columns.tolist()})
print(df.head())

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.eval()

print(model_name)

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def encode_sentences(sentences, tokenizer, model, device, batch_size=32, max_length=128):
    all_embeddings = []
    all_norms = []
    with torch.no_grad():
        for start_idx in range(0, len(sentences), batch_size):
            batch_sentences = sentences[start_idx:start_idx + batch_size]
            encoded = tokenizer(
                batch_sentences,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(device) for k, v in encoded.items()}
            outputs = model(**encoded)
            pooled = mean_pool(outputs.last_hidden_state, encoded["attention_mask"])
            normalized = F.normalize(pooled, p=2, dim=1)
            norms = torch.linalg.norm(pooled, dim=1)
            all_embeddings.append(normalized.cpu())
            all_norms.append(norms.cpu())
    embeddings = torch.cat(all_embeddings, dim=0).numpy().astype(np.float32)
    norms = torch.cat(all_norms, dim=0).numpy().astype(np.float32)
    return embeddings, norms

In [ ]:
sentences1 = df["sentence1"].tolist()
sentences2 = df["sentence2"].tolist()
labels = df["label"].to_numpy(dtype=np.float32)

emb1, emb1_norm = encode_sentences(
    sentences1,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=batch_size,
    max_length=max_length,
)

emb2, emb2_norm = encode_sentences(
    sentences2,
    tokenizer=tokenizer,
    model=model,
    device=device,
    batch_size=batch_size,
    max_length=max_length,
)

cosine_similarity = np.sum(emb1 * emb2, axis=1).astype(np.float32)
predicted_score_0_5 = (2.5 * (cosine_similarity + 1.0)).astype(np.float32)
absolute_error = np.abs(predicted_score_0_5 - labels).astype(np.float32)
squared_error = np.square(predicted_score_0_5 - labels).astype(np.float32)
signed_error = (predicted_score_0_5 - labels).astype(np.float32)
agreement_gap = np.abs(cosine_similarity - (labels / 2.5 - 1.0)).astype(np.float32)
embedding_norm_gap = np.abs(emb1_norm - emb2_norm).astype(np.float32)

results_df = df.copy()
results_df["emb1_norm"] = emb1_norm
results_df["emb2_norm"] = emb2_norm
results_df["embedding_norm_gap"] = embedding_norm_gap
results_df["cosine_similarity"] = cosine_similarity
results_df["predicted_score_0_5"] = predicted_score_0_5
results_df["absolute_error"] = absolute_error
results_df["squared_error"] = squared_error
results_df["signed_error"] = signed_error
results_df["agreement_gap"] = agreement_gap

print(results_df[["sentence1", "sentence2", "label", "emb1_norm", "emb2_norm", "cosine_similarity", "predicted_score_0_5", "absolute_error"]].head(10))

In [ ]:
pearson_corr = pearsonr(results_df["predicted_score_0_5"], results_df["label"]).statistic
spearman_corr = spearmanr(results_df["predicted_score_0_5"], results_df["label"]).statistic
mae = float(results_df["absolute_error"].mean())
rmse = float(np.sqrt(results_df["squared_error"].mean()))

label_bin_edges = [-0.001, 1.0, 2.0, 3.0, 4.0, 5.001]
label_bin_names = ["[0,1)", "[1,2)", "[2,3)", "[3,4)", "[4,5]"]
results_df["label_bin"] = pd.cut(
    results_df["label"],
    bins=label_bin_edges,
    labels=label_bin_names,
    include_lowest=True,
    right=False,
)

bin_agg = (
    results_df.groupby("label_bin", observed=False)
    .agg(
        count=("label", "size"),
        label_mean=("label", "mean"),
        pred_mean=("predicted_score_0_5", "mean"),
        cosine_mean=("cosine_similarity", "mean"),
        emb1_norm_mean=("emb1_norm", "mean"),
        emb2_norm_mean=("emb2_norm", "mean"),
        norm_gap_mean=("embedding_norm_gap", "mean"),
        mae=("absolute_error", "mean"),
        rmse=("squared_error", lambda x: float(np.sqrt(np.mean(x)))),
        signed_error_mean=("signed_error", "mean"),
    )
    .reset_index()
)

norm_summary = {
    "emb1_norm_mean": round(float(results_df["emb1_norm"].mean()), 6),
    "emb1_norm_std": round(float(results_df["emb1_norm"].std()), 6),
    "emb2_norm_mean": round(float(results_df["emb2_norm"].mean()), 6),
    "emb2_norm_std": round(float(results_df["emb2_norm"].std()), 6),
    "embedding_norm_gap_mean": round(float(results_df["embedding_norm_gap"].mean()), 6),
    "embedding_norm_gap_max": round(float(results_df["embedding_norm_gap"].max()), 6),
}

print({
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
})
print(norm_summary)
print(bin_agg)

In [ ]:
best_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "embedding_norm_gap", "label"],
        ascending=[True, True, True, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "emb1_norm", "emb2_norm", "embedding_norm_gap", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

worst_examples = (
    results_df.sort_values(
        by=["absolute_error", "agreement_gap", "embedding_norm_gap", "label"],
        ascending=[False, False, False, False],
    )
    [["sentence1", "sentence2", "label", "predicted_score_0_5", "cosine_similarity", "emb1_norm", "emb2_norm", "embedding_norm_gap", "absolute_error", "signed_error"]]
    .head(10)
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 160)
print("BEST_AGREEMENT_EXAMPLES")
print(best_examples)
print("\nWORST_AGREEMENT_EXAMPLES")
print(worst_examples)

In [ ]:
runtime_seconds = time.time() - start_time

summary = {
    "device_used": device,
    "model_name": model_name,
    "dataset_split": f"{dataset_name}/{dataset_config}/{split_name}",
    "num_examples": int(len(df)),
    "pearson_correlation": round(float(pearson_corr), 6),
    "spearman_correlation": round(float(spearman_corr), 6),
    "mae_0_5": round(mae, 6),
    "rmse_0_5": round(rmse, 6),
    "emb1_norm_mean": round(float(results_df["emb1_norm"].mean()), 6),
    "emb2_norm_mean": round(float(results_df["emb2_norm"].mean()), 6),
    "embedding_norm_gap_mean": round(float(results_df["embedding_norm_gap"].mean()), 6),
    "runtime_seconds": round(float(runtime_seconds), 2),
}

print(summary)